# Pilot Evauation using Llama Parse

Testing performance of the system developed initially on 10 randomly chosen and manually annotated papers. 

## **Select 10 Randomly Identified PDFs**

In [ ]:
# Data
path = r"D:\3. Research Main Room\Publications\1 Current Projects\Data Science and Tourism Research - Conceptual Study\DATA ANALYSIS\Data_files\DATA"

In [ ]:
from pathlib import Path

folder = Path(path)

# Only PDFs in this folder
pdfs = sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".pdf"]) #This will catch .pdf, .PDF, .Pdf, etc.

# If you want subfolders too, use this instead:
# pdfs = sorted(folder.rglob("*.pdf"))

#for i, f in enumerate(pdfs, 1):
    #print(f"{i}. {f.name}")

# Print length
print(f"Total PDFs: {len(pdfs)}")

Random Selection

In [ ]:
import random

random.seed(42)

sample_size = min(10, len(pdfs))
random_pdfs = random.sample(pdfs, k=sample_size)

print(f"Randomly selected {sample_size} PDF(s):")
for i, f in enumerate(random_pdfs, 1):
    print(f"{i}. {f.name}")

In [ ]:
import shutil

# Change this to your destination folder
destination_folder = Path(r"C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10")
destination_folder.mkdir(parents=True, exist_ok=True)

for f in random_pdfs:
    shutil.copy2(f, destination_folder / f.name)

print(f"Copied {len(random_pdfs)} file(s) to: {destination_folder}")

# llama parse pipeline


In [ ]:
paper = destination_folder / "2023 - 58.pdf"
print(f"Sample paper path: {paper}")

In [ ]:
from llama_parse import LlamaParse

parser = LlamaParse(
    result_type="markdown",   # best for LLMs
    verbose=True
)

documents = parser.load_data(str(paper))

print(documents[0].text[:1000])

In [ ]:
parser = LlamaParse(
    result_type="markdown",
    parsing_instruction="""
    Extract the document with clear section separation.
    Preserve headings like Abstract, Methodology, Results, Conclusion.
    Keep tables structured.
    """
)

In [ ]:
from llama_index.core.node_parser import MarkdownNodeParser

node_parser = MarkdownNodeParser()
nodes = node_parser.get_nodes_from_documents(documents)

In [ ]:
for i, f in enumerate(random_pdfs):
    print(f"{i}. {f.name}")

## Run data_understanding node (with indexing)
This section indexes sampled PDFs into Chroma using LlamaParse output, then runs the data understanding node for one paper.

In [ ]:
import os
import sys
from pathlib import Path

# Resolve project root so node imports + prompt paths work
cwd = Path.cwd().resolve()
project_root = cwd
while project_root != project_root.parent and not (project_root / "nodes").exists():
    project_root = project_root.parent

if not (project_root / "nodes").exists():
    raise RuntimeError("Could not locate project root containing 'nodes' folder.")

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set.")

embedding_model = "text-embedding-3-small"
collection_name = "pilot_study_10_llamaparse_node_eval"
persist_directory = str(project_root / "Pilot_Evaluation" / "Pilot_Evaluation" / "llama_parse_pilot_evaluation" / "pilot_study_node_vectors")

vectorstore = Chroma(
    collection_name=collection_name,
    embedding_function=OpenAIEmbeddings(model=embedding_model),
    persist_directory=persist_directory
)

splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)

def paper_already_indexed(vs, paper_id: str) -> bool:
    existing = vs.get(where={"paper_id": paper_id})
    return len(existing.get("ids", [])) > 0

def index_sample_with_llamaparse(pdf_paths):
    indexed, skipped = 0, 0
    for pdf_path in pdf_paths:
        paper_id = pdf_path.name  # keep consistent with node metadata filter
        if paper_already_indexed(vectorstore, paper_id):
            print(f"Skipping already indexed: {paper_id}")
            skipped += 1
            continue

        print(f"Parsing and indexing: {paper_id}")
        parsed_docs = parser.load_data(str(pdf_path))

        docs_to_add = []
        ids_to_add = []
        chunk_idx = 0

        for parsed_doc in parsed_docs:
            text = getattr(parsed_doc, "text", "") or ""
            if not text.strip():
                continue

            for chunk in splitter.split_text(text):
                doc_id = f"{paper_id}_chunk_{chunk_idx}"
                docs_to_add.append(
                    Document(
                        page_content=chunk,
                        metadata={
                            "paper_id": paper_id,
                            "filename": pdf_path.name,
                            "section_heading": None,
                            "page_no": None,
                            "modality": "text"
                        },
                    )
                )
                ids_to_add.append(doc_id)
                chunk_idx += 1

        if docs_to_add:
            vectorstore.add_documents(docs_to_add, ids=ids_to_add)
            indexed += 1
            print(f"Indexed {len(docs_to_add)} chunks for {paper_id}")
        else:
            print(f"No text extracted for {paper_id}")

    vectorstore.persist()
    print(f"Done. Indexed: {indexed}, skipped: {skipped}")

index_sample_with_llamaparse(random_pdfs)
print(f"Persist directory: {persist_directory}")

In [ ]:
for i, f in enumerate(random_pdfs):
    print(f"{i}. {f.name}")

In [ ]:
import json
from nodes.data_understanding_node import data_understanding_node

# Pick one indexed paper (filename with extension, matching metadata)
paper_id_for_eval = random_pdfs[9].name
print(f"Running data_understanding_node for: {paper_id_for_eval}")

state = {
    "paper_id": paper_id_for_eval,
    "dsrp_outputs": {},
    "collection_name": collection_name,
    "persist_directory": persist_directory,
    "embedding_model": embedding_model,
    "llm_model": "gpt-4o-mini",
}

result = data_understanding_node(state)
print(json.dumps(result["dsrp_outputs"].get("data_understanding", {}), indent=2, ensure_ascii=False))

In [ ]:
import os
import re
import json
import importlib
from pathlib import Path
import pandas as pd
import nodes.data_understanding_node as data_understanding_module
from utils.dsrp_state import DSRPState

# Reload module to avoid stale function in notebook kernel
importlib.reload(data_understanding_module)
data_understanding_node = data_understanding_module.data_understanding_node

# Try different models here (examples: "gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1")
LLM_MODEL = "gpt-4o-mini"

# Keep this cell self-contained so GT comparison always works
DIMENSIONS = ["data_category", "data_format", "data_characteristics"]

SYNONYM_MAP = {
    "data_category": {
        "ugc": "UGC",
        "device": "Device",
        "transaction": "Transaction",
        "survey/statistical": "Survey/Statistical",
        "survey / statistical": "Survey/Statistical",
    },
    "data_format": {
        "structured": "Structured",
        "semi-structured": "Semi-Structured",
        "semistructured": "Semi-Structured",
        "unstructured": "Unstructured",
    },
    "data_characteristics": {
        "temporal": "Temporal",
        "spatial": "Spatial",
        "textual": "Textual",
        "visual": "Visual",
        "networked": "Networked",
    },
}

def normalize_text(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value).strip()
    return text if text else None

def normalize_paper_id_key(value):
    """Normalize for benchmark matching: lowercase and strip optional .pdf suffix."""
    text = normalize_text(value)
    if not text:
        return None
    text = text.lower()
    if text.endswith(".pdf"):
        text = text[:-4]
    return text.strip()

def ensure_pdf_suffix(value):
    """Normalize for node retrieval: ensure .pdf suffix exists."""
    text = normalize_text(value)
    if not text:
        return None
    if not text.lower().endswith(".pdf"):
        return f"{text}.pdf"
    return text

def clean_label_token(token):
    t = token.strip().lower()
    t = re.sub(r"\s+", " ", t)
    t = t.replace(" / ", "/")
    return t

def parse_multilabel(value, dimension):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    if isinstance(value, list):
        raw_items = value
    else:
        text = str(value).strip()
        if not text:
            return []
        raw_items = [x for part in text.split(";") for x in part.split(",")]

    mapped = []
    for item in raw_items:
        token = clean_label_token(str(item))
        canonical = SYNONYM_MAP[dimension].get(token)
        if canonical:
            mapped.append(canonical)

    seen = set()
    out = []
    for label in mapped:
        if label not in seen:
            seen.add(label)
            out.append(label)
    return out

def get_nested(data, path, default=None):
    current = data
    for key in path.split("."):
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(project_root)

VECTOR_DB_PATH = Path(persist_directory)
COLLECTION_NAME = collection_name
EMBEDDING_MODEL = embedding_model

# Locate benchmark file (supports both Pilot_Evaluation and nested Pilot_Evaluation/Pilot_Evaluation layouts)
benchmark_candidates = [
    project_root / "Pilot_Evaluation" / "DATA_sample_10" / "Data Science Research Process (DSRP) Framework.csv",
    project_root / "Pilot_Evaluation" / "Pilot_Evaluation" / "DATA_sample_10" / "Data Science Research Process (DSRP) Framework.csv",
]
BENCHMARK_FILE = next((p for p in benchmark_candidates if p.exists()), None)
if BENCHMARK_FILE is None:
    raise FileNotFoundError("Could not find benchmark CSV for ground-truth comparison.")

benchmark_df = pd.read_csv(BENCHMARK_FILE)
benchmark_df.columns = benchmark_df.columns.str.strip()
comment_cols = [
    c for c in benchmark_df.columns
    if "Any Comments" in c or "Any comments" in c
]
benchmark_df_clean = benchmark_df.drop(columns=comment_cols, errors="ignore")

required_cols = [
    "Paper ID",
    "Gatekeeper",
    "data_category",
    "data_format",
    "data_characteristics",
]
missing = [c for c in required_cols if c not in benchmark_df_clean.columns]
if missing:
    raise ValueError(f"Missing required benchmark columns: {missing}")

# Normalize benchmark IDs for matching (no .pdf suffix)
benchmark_df_clean["paper_id_norm"] = benchmark_df_clean["Paper ID"].apply(normalize_paper_id_key)

# Prefer prior incorrect list if available; otherwise evaluate sampled papers
if "wrong_paper_ids" in globals() and len(wrong_paper_ids):
    paper_ids_to_retest = [normalize_text(x) for x in wrong_paper_ids if normalize_text(x)]
    print(f"Using wrong_paper_ids from notebook: {len(paper_ids_to_retest)} papers")
else:
    paper_ids_to_retest = [normalize_text(p.name) for p in random_pdfs if normalize_text(p.name)]
    print(f"Using sampled papers from random_pdfs: {len(paper_ids_to_retest)} papers")

retest_results = []

for paper_id_raw in paper_ids_to_retest:
    paper_id_for_node = ensure_pdf_suffix(paper_id_raw)
    paper_id_key = normalize_paper_id_key(paper_id_raw)

    print(f"Re-running data_understanding node for {paper_id_for_node} with {LLM_MODEL}...")
    state = DSRPState(
        paper_id=paper_id_for_node,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
        llm_model=LLM_MODEL,
    )

    try:
        out = data_understanding_node(state)
        outputs = get_nested(out, "dsrp_outputs", default={})
        du = outputs.get("data_understanding", {})
        pre_audit = outputs.get("data_understanding_pre_audit", {})
        classifier_outputs = pre_audit.get("classification_outputs", {})

        print("  Pre-audit classifier outputs:")
        for dim_key in DIMENSIONS:
            print(f"    {dim_key}: {classifier_outputs.get(dim_key, {})}")

        benchmark_row = benchmark_df_clean.loc[benchmark_df_clean["paper_id_norm"] == paper_id_key]
        if benchmark_row.empty:
            retest_results.append({
                "Paper ID": paper_id_raw,
                "Paper ID (node)": paper_id_for_node,
                "Model": LLM_MODEL,
                "Error": "Paper ID not found in benchmark CSV",
            })
            print("  Error: Paper ID not found in benchmark CSV")
            continue

        row = benchmark_row.iloc[0]
        gatekeeper_label = (normalize_text(row.get("Gatekeeper")) or "").lower()

        retest_row = {
            "Paper ID": paper_id_raw,
            "Paper ID (node)": paper_id_for_node,
            "Model": LLM_MODEL,
            "Gatekeeper": gatekeeper_label,
        }
        match_flags = []

        for dim in DIMENSIONS:
            gt_labels = sorted(set(parse_multilabel(row.get(dim), dim)))
            pred_labels = sorted(set(parse_multilabel(du.get(dim), dim)))
            dim_match = gt_labels == pred_labels
            match_flags.append(dim_match)

            retest_row[f"GT_{dim}"] = gt_labels
            retest_row[f"New_Agent_{dim}"] = pred_labels
            retest_row[f"Match_{dim}"] = "MATCH" if dim_match else "MISMATCH"

        retest_row["Match_All_3"] = "MATCH" if all(match_flags) else "MISMATCH"
        retest_results.append(retest_row)

    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id_raw,
            "Paper ID (node)": paper_id_for_node,
            "Model": LLM_MODEL,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    print("\nRE-TEST RESULTS (WITH GROUND-TRUTH COMPARISON)")
    print(retest_df.to_string(index=False))

    if "Match_All_3" in retest_df.columns:
        valid = retest_df[retest_df["Match_All_3"].notna()]
        if len(valid):
            exact = int((valid["Match_All_3"] == "MATCH").sum())
            total = len(valid)
            print(f"\nOverall exact agreement across all 3 dimensions: {exact}/{total} ({(100 * exact / total):.1f}%)")
else:
    print("No papers to re-test.")

# No Significant Improvement!